# Optuna Hyperparameter Tuning per il Clustering

Questo notebook esegue la ricerca degli iperparametri ottimali (Optuna) per HDBSCAN (ed eventualmente UMAP). 
Al fine di massimizzare l'efficienza, l'ottimizzazione viene eseguita su un **campione stratificato** (subsample) degli embeddings (es. 15.000 righe), invece che sull'intero dataset massivo.

In [1]:
import sys
import os
from pathlib import Path

# Aggiungiamo src al path per poter importare i moduli
sys.path.append(os.path.abspath(os.path.join('..', '..')))

import numpy as np
import optuna
import optuna.visualization as vis
import matplotlib.pyplot as plt
from src.utils.optuna_clustering import run_clustering_optimization

# Setup logging
import logging
logging.basicConfig(level=logging.INFO)

### 1. Caricamento del dataset esteso e campionamento

In [2]:
# Puntiamo al dataset esteso per fare un test più veritiero come richiesto
EMBEDDINGS_PATH = "../../data/embeddings/email_embeddings.npy"

print(f"Caricamento degli embeddings da: {EMBEDDINGS_PATH}")
total_embeddings = np.load(EMBEDDINGS_PATH)

SAMPLE_SIZE = 15000
if len(total_embeddings) > SAMPLE_SIZE:
    print(f"Campionamento di {SAMPLE_SIZE} vettori da un totale di {len(total_embeddings)}...")
    # Assicuriamoci di campionare sempre le stesse righe ad ogni avvio del notebook (seed fisso)
    np.random.seed(42)
    indices = np.random.choice(len(total_embeddings), size=SAMPLE_SIZE, replace=False)
    sample_embeddings = total_embeddings[indices]
else:
    sample_embeddings = total_embeddings

print(f"Shape finale del campione da ottimizzare: {sample_embeddings.shape}")

Caricamento degli embeddings da: ../../data/embeddings/email_embeddings.npy
Campionamento di 15000 vettori da un totale di 1757624...
Shape finale del campione da ottimizzare: (15000, 384)


### 2. Esecuzione dell'ottimizzazione (TPE o CMA-ES)

In [3]:
# Puoi cambiare sampler_type in 'CMA-ES' per testare l'algoritmo genetico/evolutivo
# Impostando use_umap=True verranno ottimizzati anche i parametri di UMAP
study = run_clustering_optimization(
    embeddings=sample_embeddings, 
    n_trials=50, # Inizia con 20, poi magari alza a 50 o 100
    use_umap=True, 
    sampler_type='CMA-ES',
    penalty_weight=0.5 # Aumenta per penalizzare maggiormente i cluster con troppo rumore/outliers
)

INFO:src.utils.optuna_clustering:Avvio ottimizzazione Optuna con 50 trials, sampler=CMA-ES, use_umap=True
[I 2026-06-19 09:47:03,567] A new study created in memory with name: no-name-b0f81b05-e454-4205-bd0c-cbf8f565b1fb


  0%|          | 0/50 [00:00<?, ?it/s]

/home/filippo/Scrivania/pizza-cluster/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[I 2026-06-19 09:47:29,868] Trial 0 finished with value: 0.3678062174161275 and parameters: {'n_neighbors': 44, 'n_components': 15, 'min_cluster_size': 149, 'min_samples': 122}. Best is trial 0 with value: 0.3678062174161275.


/home/filippo/Scrivania/pizza-cluster/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


...

[I 2026-06-19 09:57:06,932] Trial 47 finished with value: 0.3936986922264099 and parameters: {'n_neighbors': 46, 'n_components': 12, 'min_cluster_size': 91, 'min_samples': 63}. Best is trial 27 with value: 0.4087649949709574.


/home/filippo/Scrivania/pizza-cluster/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[I 2026-06-19 09:57:16,413] Trial 48 finished with value: 0.3539644276301066 and parameters: {'n_neighbors': 27, 'n_components': 12, 'min_cluster_size': 65, 'min_samples': 17}. Best is trial 27 with value: 0.4087649949709574.


/home/filippo/Scrivania/pizza-cluster/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
INFO:src.utils.optuna_clustering:Miglior trial: 27
INFO:src.utils.optuna_clustering:Migliori parametri: {'n_neighbors': 51, 'n_components': 12, 'min_cluster_size': 71, 'min_samples': 100}
INFO:src.utils.optuna_clustering:Miglior fitness: 0.4087649949709574


[I 2026-06-19 09:57:27,064] Trial 49 finished with value: 0.4005157941500346 and parameters: {'n_neighbors': 35, 'n_components': 10, 'min_cluster_size': 101, 'min_samples': 104}. Best is trial 27 with value: 0.4087649949709574.


### 3. Analisi dei Risultati

In [4]:
if len(study.best_trials) > 0:
    print("Migliori iperparametri trovati:")
    for key, value in study.best_params.items():
        print(f"  {key}: {value}")
    
    best_trial = study.best_trial
    print(f"\nFitness Score Ottenuto: {best_trial.value:.4f}")
    print(f"  - Cluster trovati: {best_trial.user_attrs.get('n_clusters', 'N/A')}")
    print(f"  - Silhouette Score: {best_trial.user_attrs.get('silhouette', 'N/A'):.4f}")
    print(f"  - Rumore (Outliers): {best_trial.user_attrs.get('noise_ratio', 'N/A'):.2%}")
else:
    print("Nessun trial valido completato.")

Migliori iperparametri trovati:
  n_neighbors: 51
  n_components: 12
  min_cluster_size: 71
  min_samples: 100

Fitness Score Ottenuto: 0.4088
  - Cluster trovati: 21
  - Silhouette Score: 0.6354
  - Rumore (Outliers): 45.33%


In [5]:
# Visualizza la storia dell'ottimizzazione
vis.plot_optimization_history(study)

In [6]:
# Visualizza l'importanza degli iperparametri (se ne hai ottimizzato più di 1)
try:
    fig = vis.plot_param_importances(study)
    fig.show()
except Exception as e:
    print("Impossibile generare il grafico dell'importanza (serve più varianza):", e)